# Build segmentation dataset — 12-group structure

New dataset in a **new folder** `transformed_data/`, per the plan: original +
derived features following the requested PLAYER tree.

**Outputs**
* `transformed_data/player_dataset.csv` — **100,000 rows x 70 columns** (player_id key + 69 features)
* `transformed_data/dataset_dictionary.csv` — every structure entry with kind
  (`key / raw / derived / shared / unavailable / excluded`) + formula

**Decisions (from Q&A)**

| decision | choice |
|---|---|
| Future / better-data columns | only the 3 derivable ones included: `GGR_per_day`, `deposits_per_month`, `bets_per_month`; the other 17 are listed in the dictionary as `unavailable` (no source field) |
| Raw monetary totals | **transformed only**: `monetary_scale = log1p(total_deposits)` and `betting_intensity = total_bets/total_deposits` replace `total_deposits`/`total_bets` (raw totals still drive the derived rates, they are just not kept as columns) |
| `win_rate` | both: `win_rate = total_wins/total_bets` **and** `winning_rate = total_wins/tenure_months` |
| Employer fields | all `employer_*` excluded as PII |
| Engagement derived | `logins_per_active_day = total_logins/account_lifetime_days` (active_days unavailable → lifetime proxy); `login_frequency = total_logins/tenure_months` (alias of `login_rate`) |
| PII | `player_id` kept as the table key; group-12 fields dropped |

**Aliases kept under both names (structure lists them separately):**
`login_frequency` ≡ `login_rate` · `deposits_per_month` ≡ `deposit_rate` ·
`bets_per_month` ≡ `betting_rate` · `betting_to_deposit_ratio` ≡ `betting_intensity`

**Conventions** (same as `feature_engineering.ipynb`): `REFERENCE_DATE =
max(loaded_at)` = 2025-11-13; `tenure_months = tenure_days / 30.4375`;
`safe_div` → `NaN` on zero denominators; age kept only in [18, 100];
`betting_intensity` uses the clustering-matrix convention (0/0 → 0,
deposit-free betting → `NaN`).

**Request 2 — recency / frequency / intensity / product / temporal / value audit**

Only one aggregated player-level file exists in the project (the 77-column CSV);
there is no event / session / bet / bonus / product data, so most of the requested
set cannot be computed. Of the 24 listed features:

* **already in dataset:** `GGR`
* **built now:** `margin = GGR / total_bets` (group 3);
  `registration_weekend` and `registration_daypart` (group 1) — the only honest
  proxies for *weekday/weekend* and *day/night*, since `registration_date` is the
  only player-action timestamp that carries a time-of-day
* **impossible as specified (20)** — need event-level data, all listed in the
  dictionary as `unavailable`: recency (`days_since_last_login`, `days_since_last_bet`,
  `days_since_last_deposit`, `days_since_last_withdrawal`), frequency (`active_days`,
  `betting_days`, `deposit_count`, `withdrawal_count`, `sessions`), intensity
  (`average_bet`, `median_bet`, `bets_per_session`, `turnover_per_active_day`),
  product (`casino_vs_sports`, `game_count`, `game_diversity`, `preferred_product`),
  temporal (`activity trend`; *weekday/weekend* and *day/night* rows are also
  `unavailable` at activity level — the group-1 proxies stand in for them), value
  (`NGR`, `bonus_cost` — bonus cost is unknown).

**Request 3 — de-skew + margin removal**

All 19 numeric features with |skew| > 3 are converted **in place** (originals dropped — transformed-only, consistent with `monetary_scale`): non-negative →
`log1p(x)`; features carrying negatives → `slog1p(x) = sign(x)*log1p(|x|)` (`slog1p_time_to_first_deposit`, `slog1p_net_deposits`, `slog1p_GGR`,
`slog1p_GGR_per_day`). New names `log1p_<feature>` / `slog1p_<feature>`. **`margin` removed** (identical to `1 - win_rate`, r = -1). Outputs rebuild:
`transformed_data/player_dataset.csv` (100,000 x 72) and `dataset_dictionary.csv` (118 rows; margin kept with kind `removed`).

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

SRC = "Data Science Task Data.xlsx - Result 1.csv"
OUT = "transformed_data"
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(SRC, low_memory=False)
DATE_COLS = ["registration_date", "modified_at", "closed_at", "birth_date",
             "first_deposit_day", "loaded_at", "vip_status_updated_at"]
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

REFERENCE_DATE = df["loaded_at"].max()
tenure_days = (REFERENCE_DATE - df["registration_date"]).dt.days
tenure_months = tenure_days / 30.4375


def safe_div(num, den):
    return num.div(den).replace([np.inf, -np.inf], np.nan)


print(f"source: {df.shape} | REFERENCE_DATE: {REFERENCE_DATE}")

source: (100000, 77) | REFERENCE_DATE: 2025-11-13 08:03:42


In [2]:
F = {}

# ---- 1. lifecycle --------------------------------------------------------
age_raw = (REFERENCE_DATE - df["birth_date"]).dt.days / 365.25
F["age"] = age_raw.where(age_raw.between(18, 100))
F["tenure_days"] = tenure_days
F["time_to_first_deposit"] = (df["first_deposit_day"].dt.normalize() -
                              df["registration_date"].dt.normalize()).dt.days
F["account_lifetime_days"] = ((df["closed_at"] - df["registration_date"]).dt.days
                              .fillna(tenure_days))
TERMINAL = ["Closed", "Suspended", "SelfExclusion", "Timeout", "Duplicated"]
F["is_closed"] = df["closed_at"].notna() | df["status"].isin(TERMINAL)
F["is_active"] = df["status"].eq("Approved")
for c in ["registration_date", "first_deposit_day", "closed_at", "status"]:
    F[c] = df[c]

# ---- 2. engagement -------------------------------------------------------
F["total_logins"] = df["total_logins"]
F["login_rate"] = safe_div(df["total_logins"], tenure_months)
F["logins_per_active_day"] = safe_div(df["total_logins"], F["account_lifetime_days"])
F["login_frequency"] = F["login_rate"]

# ---- 3. economic / value -------------------------------------------------
F["monetary_scale"] = np.log1p(df["total_deposits"])
F["total_withdrawals"] = df["total_withdrawals"]
bi = df["total_bets"] / df["total_deposits"]
bi = bi.replace([np.inf, -np.inf], np.nan)
bi[df["total_deposits"].eq(0) & df["total_bets"].eq(0)] = 0.0
F["betting_intensity"] = bi
F["total_wins"] = df["total_wins"]
F["net_deposits"] = df["total_deposits"] - df["total_withdrawals"]
both = df["total_bets"].notna() & df["total_wins"].notna()
F["GGR"] = (df["total_bets"] - df["total_wins"]).where(both)
F["deposit_rate"] = safe_div(df["total_deposits"], tenure_months)
F["withdrawal_rate"] = safe_div(df["total_withdrawals"], tenure_months)
F["betting_rate"] = safe_div(df["total_bets"], tenure_months)
F["win_rate"] = safe_div(df["total_wins"], df["total_bets"])
F["winning_rate"] = safe_div(df["total_wins"], tenure_months)
F["betting_to_deposit_ratio"] = F["betting_intensity"]
F["GGR_per_day"] = safe_div(F["GGR"], F["account_lifetime_days"])
F["deposits_per_month"] = F["deposit_rate"]
F["bets_per_month"] = F["betting_rate"]

# ---- request 2: value + registration-time proxies -------------------------
F["margin"] = safe_div(F["GGR"], df["total_bets"])
reg_ok = df["registration_date"].notna()
F["registration_weekend"] = (df["registration_date"].dt.dayofweek >= 5).where(reg_ok)
hour = df["registration_date"].dt.hour
F["registration_daypart"] = pd.Series(
    np.select([hour.between(6, 11), hour.between(12, 17), hour.between(18, 22)],
              ["morning", "afternoon", "evening"], default="night"),
    index=df.index).where(reg_ok)

# ---- 4. acquisition ------------------------------------------------------
F["referral_flag"] = (df["referral_code"].notna() | df["referrer_player_id"].notna())
for c in ["traffic_source", "sub_traffic_source", "registration_platform",
          "utm_source", "utm_medium", "utm_campaign", "utm_content", "utm_term",
          "ad_campaign_id", "ad_group_id", "ad_keyword_id",
          "facebook_click_id", "google_click_id", "microsoft_click_id",
          "tiktok_click_id", "d_click_id", "referral_code", "referrer_player_id"]:
    F[c] = df[c]

# ---- 5-11 passthrough ----------------------------------------------------
for c in ["gender", "country", "city", "occupation",
          "vip_level", "portfolio_vip_status", "player_vip_account_manager",
          "vip_status_updated_at", "operational_profit_segment",
          "max_profit_segment_rank",
          "aml_risk_rating", "is_pep", "has_sanctions",
          "ever_self_excluded", "is_bonus_abuser",
          "gaming_level", "allow_sms", "allow_email", "has_mfa_enabled",
          "modified_at", "loaded_at"]:
    F[c] = df[c]

# request 3: de-skew numeric features with |skew| > 3 (margin already dropped)
F.pop("margin", None)
CONV = {}
def _slog1p(x):
    return np.sign(x) * np.log1p(np.abs(x))
for _c in list(F):
    _s = F[_c]
    if _s.dtype == "bool" or not pd.api.types.is_numeric_dtype(_s) or _s.dropna().empty:
        continue
    _sk = _s.dropna().skew()
    if abs(_sk) <= 3:
        continue
    _prefix = "log1p" if _s.dropna().min() >= 0 else "slog1p"
    _new = f"{_prefix}_{_c}"
    F[_new] = np.log1p(F[_c]) if _prefix == "log1p" else _slog1p(F[_c])
    CONV[_new] = (_c, _sk, _prefix)
print(f"de-skewed {len(CONV)} features: {', '.join(sorted(CONV))}")

print(f"derived/passthrough series ready: {len(F)}")

de-skewed 19 features: log1p_bets_per_month, log1p_betting_intensity, log1p_betting_rate, log1p_betting_to_deposit_ratio, log1p_deposit_rate, log1p_deposits_per_month, log1p_login_frequency, log1p_login_rate, log1p_logins_per_active_day, log1p_total_logins, log1p_total_wins, log1p_total_withdrawals, log1p_win_rate, log1p_winning_rate, log1p_withdrawal_rate, slog1p_GGR, slog1p_GGR_per_day, slog1p_net_deposits, slog1p_time_to_first_deposit
derived/passthrough series ready: 90


In [3]:
GROUPS = [
    ("1. Lifecycle",
     ["age", "tenure_days", "slog1p_time_to_first_deposit", "account_lifetime_days",
      "is_closed", "is_active", "registration_date", "first_deposit_day",
      "closed_at", "status",
      "registration_weekend", "registration_daypart"]),
    ("2. Engagement / Activity",
     ["log1p_total_logins", "log1p_login_rate", "log1p_logins_per_active_day",
      "log1p_login_frequency"]),
    ("3. Economic / Value",
     ["monetary_scale", "log1p_total_withdrawals", "log1p_betting_intensity",
      "log1p_total_wins", "slog1p_net_deposits", "slog1p_GGR", "log1p_deposit_rate",
      "log1p_withdrawal_rate", "log1p_betting_rate", "log1p_win_rate",
      "log1p_winning_rate", "log1p_betting_to_deposit_ratio", "slog1p_GGR_per_day",
      "log1p_deposits_per_month", "log1p_bets_per_month"]),
    ("4. Acquisition / Marketing",
     ["traffic_source", "sub_traffic_source", "registration_platform",
      "referral_flag", "utm_source", "utm_medium", "utm_campaign",
      "utm_content", "utm_term", "ad_campaign_id", "ad_group_id",
      "ad_keyword_id", "facebook_click_id", "google_click_id",
      "microsoft_click_id", "tiktok_click_id", "d_click_id",
      "referral_code", "referrer_player_id"]),
    ("5. Demographics",
     ["gender", "country", "city", "occupation"]),
    ("6. VIP / Commercial Context",
     ["vip_level", "portfolio_vip_status", "player_vip_account_manager",
      "vip_status_updated_at", "operational_profit_segment",
      "max_profit_segment_rank"]),
    ("7. AML / Compliance", ["aml_risk_rating", "is_pep", "has_sanctions"]),
    ("8. Responsible Gambling", ["ever_self_excluded"]),
    ("9. Fraud / Abuse / Integrity", ["is_bonus_abuser"]),
    ("10. Account / Security / Preferences",
     ["gaming_level", "allow_sms", "allow_email", "has_mfa_enabled"]),
    ("11. Technical / System Metadata", ["modified_at", "loaded_at"]),
]

cols = ["player_id"] + [c for _, cs in GROUPS for c in cs]
assert len(cols) == len(set(cols)), "duplicate column"
player_dataset = pd.concat([df[["player_id"]]] +
                           [pd.DataFrame(F, index=df.index)[cs] for _, cs in GROUPS],
                           axis=1)[cols]

assert player_dataset.shape == (100000, 72), player_dataset.shape
print(f"player_dataset: {player_dataset.shape}")
for g, cs in GROUPS:
    print(f"  {g:<35} {len(cs):>3} cols")

player_dataset: (100000, 72)
  1. Lifecycle                         12 cols
  2. Engagement / Activity              4 cols
  3. Economic / Value                  15 cols
  4. Acquisition / Marketing           19 cols
  5. Demographics                       4 cols
  6. VIP / Commercial Context           6 cols
  7. AML / Compliance                   3 cols
  8. Responsible Gambling               1 cols
  9. Fraud / Abuse / Integrity          1 cols
  10. Account / Security / Preferences   4 cols
  11. Technical / System Metadata       2 cols


In [4]:
D = []


def add(group, feature, kind, formula):
    D.append({"group": group, "feature": feature, "kind": kind, "formula": formula})


G = "1. Lifecycle"
add(G, "age", "derived", "(max(loaded_at) - birth_date)/365.25, kept in [18,100]")
add(G, "tenure_days", "derived", "(max(loaded_at) - registration_date).days")
add(G, "time_to_first_deposit", "derived", "first_deposit_day - registration_date (calendar days)")
add(G, "account_lifetime_days", "derived", "(closed_at - registration_date).days, else tenure_days")
add(G, "is_closed", "derived", "closed_at present or status in terminal states")
add(G, "is_active", "derived", "status == 'Approved'")
for c in ["registration_date", "first_deposit_day", "closed_at", "status"]:
    add(G, c, "raw", "source column")
add(G, "registration_weekend", "derived",
    "registration_date falls on Sat/Sun (registration-time proxy for activity weekday/weekend)")
add(G, "registration_daypart", "derived",
    "hour of registration_date -> morning(6-11)/afternoon(12-17)/evening(18-22)/night (registration-time proxy for day/night)")

G = "2. Engagement / Activity"
add(G, "total_logins", "raw", "source column")
add(G, "login_rate", "derived", "total_logins / tenure_months")
add(G, "logins_per_active_day", "derived",
    "total_logins / account_lifetime_days (active_days unavailable -> lifetime proxy)")
add(G, "login_frequency", "derived", "alias of login_rate")
for c in ["active_days", "sessions", "sessions_per_day", "average_session_duration",
          "median_session_duration", "days_since_last_login", "days_since_last_bet",
          "betting_frequency", "game_count", "game_category_mix"]:
    add(G, c, "unavailable", "no such field in source data")
for c in ["days_since_last_deposit", "days_since_last_withdrawal", "betting_days",
          "casino_vs_sports", "game_diversity", "preferred_product"]:
    add(G, c, "unavailable", "no event-level data in source")
add(G, "weekday/weekend", "unavailable",
    "needs activity dates; registration_weekend (group 1) is the registration-time proxy")
add(G, "day/night", "unavailable",
    "needs activity timestamps; registration_daypart (group 1) is the registration-time proxy")
add(G, "activity trend", "unavailable", "needs time series of player activity")

G = "3. Economic / Value"
add(G, "monetary_scale", "derived", "log1p(total_deposits); replaces total_deposits")
add(G, "total_withdrawals", "raw", "source column")
add(G, "betting_intensity", "derived",
    "total_bets / total_deposits (0/0 -> 0, deposit-free betting -> NaN); replaces total_bets")
add(G, "total_wins", "raw", "source column")
add(G, "net_deposits", "derived", "total_deposits - total_withdrawals")
add(G, "GGR", "derived", "total_bets - total_wins (both present)")
add(G, "margin", "derived", "GGR / total_bets (hold %)")
add(G, "deposit_rate", "derived", "total_deposits / tenure_months")
add(G, "withdrawal_rate", "derived", "total_withdrawals / tenure_months")
add(G, "betting_rate", "derived", "total_bets / tenure_months")
add(G, "win_rate", "derived", "total_wins / total_bets (player win %)")
add(G, "winning_rate", "derived", "total_wins / tenure_months")
add(G, "betting_to_deposit_ratio", "derived", "same formula as betting_intensity")
add(G, "GGR_per_day", "derived", "GGR / account_lifetime_days")
add(G, "deposits_per_month", "derived", "alias of deposit_rate")
add(G, "bets_per_month", "derived", "alias of betting_rate")
for c in ["NGR", "bonus_cost", "bonus_to_deposit_ratio", "average_bet",
          "median_bet", "NGR_per_day", "profitability"]:
    add(G, c, "unavailable", "no such field in source data")
for c in ["deposit_count", "withdrawal_count", "bets_per_session", "turnover_per_active_day"]:
    add(G, c, "unavailable", "no event-level data in source")

G = "4. Acquisition / Marketing"
for c in ["traffic_source", "sub_traffic_source", "registration_platform"]:
    add(G, c, "raw", "source column")
add(G, "referral_flag", "derived", "referral_code or referrer_player_id present")
for c in ["utm_source", "utm_medium", "utm_campaign", "utm_content", "utm_term",
          "ad_campaign_id", "ad_group_id", "ad_keyword_id",
          "facebook_click_id", "google_click_id", "microsoft_click_id",
          "tiktok_click_id", "d_click_id", "referral_code", "referrer_player_id"]:
    add(G, c, "raw", "source column")

G = "5. Demographics"
add(G, "age", "shared", "same column as group 1")
for c in ["gender", "country", "city", "occupation"]:
    add(G, c, "raw", "source column")
for c in ["employer_name", "employer_city", "employer_country",
          "employer_postal_code", "employer_phone_number"]:
    add(G, c, "excluded", "PII - excluded by decision")

G = "6. VIP / Commercial Context"
for c in ["vip_level", "portfolio_vip_status", "player_vip_account_manager",
          "vip_status_updated_at", "operational_profit_segment",
          "max_profit_segment_rank"]:
    add(G, c, "raw", "source column")

G = "7. AML / Compliance"
for c in ["aml_risk_rating", "is_pep", "has_sanctions"]:
    add(G, c, "raw", "source column")

G = "8. Responsible Gambling"
add(G, "ever_self_excluded", "raw", "source column")

G = "9. Fraud / Abuse / Integrity"
add(G, "is_bonus_abuser", "raw", "source column")

G = "10. Account / Security / Preferences"
for c in ["gaming_level", "allow_sms", "allow_email", "has_mfa_enabled"]:
    add(G, c, "raw", "source column")

G = "11. Technical / System Metadata"
for c in ["modified_at", "loaded_at"]:
    add(G, c, "raw", "source column")

G = "12. Identification / PII - EXCLUDE"
add(G, "player_id", "key", "retained as table key (excluded from features)")
for c in ["email", "phone_number", "first_name", "middle_name", "last_name",
          "employer_name", "employer_phone_number", "employer_postal_code"]:
    add(G, c, "excluded", "PII - not carried into the dataset")
add(G, "birth_date", "excluded", "identifier - used to derive age only")

dictionary = pd.DataFrame(D, columns=["group", "feature", "kind", "formula"])
print(f"dictionary: {len(dictionary)} rows -> "
      f"{dictionary['kind'].value_counts().to_dict()}")

seen = set()
print("\nPLAYER")
for gname, cs in GROUPS + [("12. Identification / PII - EXCLUDE", [])]:
    rows = dictionary[dictionary.group == gname]
    print(f"├── {gname}  [{len([r for _, r in rows.iterrows() if r['kind'] in ('raw','derived','key','shared')])} in dataset]")
    for i, (_, r) in enumerate(rows.iterrows()):
        branch = "└──" if i == len(rows) - 1 else "├──"
        marker = {"raw": " ", "derived": "*", "key": "@", "shared": "=",
                  "unavailable": "x", "excluded": "X", "removed": "-"}[r.kind]
        dup = " (shared)" if r.feature in seen and r.kind == "shared" else ""
        seen.add(r.feature)
        print(f"│   {branch} {r.feature:<32} {marker} {r.kind}{dup}")

# request 3: rename converted features in the dictionary, mark margin removed
for _new, (_old, _sk, _p) in CONV.items():
    _mk = dictionary["feature"].eq(_old)
    dictionary.loc[_mk, "feature"] = _new
    dictionary.loc[_mk, "formula"] = (
        f"{_p} applied to: {dictionary.loc[_mk, 'formula'].iloc[0]}  "
        f"[converted from {_old}, skew {_sk:.1f}]")
_mmk = dictionary["feature"].eq("margin")
dictionary.loc[_mmk, "kind"] = "removed"
dictionary.loc[_mmk, "formula"] = "redundant: margin = 1 - win_rate (r = -1); removed"

player_dataset.to_csv(f"{OUT}/player_dataset.csv", index=False)
dictionary.to_csv(f"{OUT}/dataset_dictionary.csv", index=False)
print(f"\n[saved] {OUT}/player_dataset.csv {player_dataset.shape}")
print(f"[saved] {OUT}/dataset_dictionary.csv {dictionary.shape}")

dictionary: 118 rows -> {'raw': 46, 'unavailable': 30, 'derived': 26, 'excluded': 14, 'shared': 1, 'key': 1}

PLAYER
├── 1. Lifecycle  [12 in dataset]
│   ├── age                              * derived
│   ├── tenure_days                      * derived
│   ├── time_to_first_deposit            * derived
│   ├── account_lifetime_days            * derived
│   ├── is_closed                        * derived
│   ├── is_active                        * derived
│   ├── registration_date                  raw
│   ├── first_deposit_day                  raw
│   ├── closed_at                          raw
│   ├── status                             raw
│   ├── registration_weekend             * derived
│   └── registration_daypart             * derived
├── 2. Engagement / Activity  [4 in dataset]
│   ├── total_logins                       raw
│   ├── login_rate                       * derived
│   ├── logins_per_active_day            * derived
│   ├── login_frequency                  * derived
│   ├── 


[saved] transformed_data/player_dataset.csv (100000, 72)
[saved] transformed_data/dataset_dictionary.csv (118, 4)


In [5]:
# ---- QA ------------------------------------------------------------------
num = player_dataset.select_dtypes("number").columns
arr = player_dataset[num].to_numpy(dtype=float, na_value=np.nan)
print(f"rows x cols      : {player_dataset.shape}")
print(f"unique columns   : {player_dataset.columns.is_unique}")
print(f"inf values       : {int(np.isinf(arr).sum())}  (must be 0)")

alias_checks = {
    "log1p_login_frequency == log1p_login_rate": player_dataset.log1p_login_frequency.equals(player_dataset.log1p_login_rate),
    "log1p_deposits_per_month == log1p_deposit_rate": player_dataset.log1p_deposits_per_month.equals(player_dataset.log1p_deposit_rate),
    "log1p_bets_per_month == log1p_betting_rate": player_dataset.log1p_bets_per_month.equals(player_dataset.log1p_betting_rate),
    "log1p_betting_to_deposit_ratio == log1p_betting_intensity": player_dataset.log1p_betting_to_deposit_ratio.equals(player_dataset.log1p_betting_intensity),
}
print()
for k, v in alias_checks.items():
    print(f"  {k:<55} {v}")

expm1 = np.expm1(player_dataset.monetary_scale)
print("  expm1(monetary_scale) == total_deposits            ",
      bool(np.allclose(expm1.dropna(), df.total_deposits.loc[expm1.notna()])))

derived_cols = dictionary.loc[dictionary.kind == "derived", "feature"]
derived_cols = [c for c in derived_cols.unique() if c in player_dataset.columns]
miss = (player_dataset[derived_cols].isna().mean() * 100).round(2)
miss = miss[miss > 0].sort_values(ascending=False)
print("\nmissing % of derived columns:")
print(miss.to_string() if len(miss) else "  (none)")

assert "margin" not in player_dataset.columns, "margin must be removed"
print("\nconverted features - skew before -> after:")
_ct = pd.DataFrame([{"new feature": n, "converted from": o,
                     "skew before": round(s, 1),
                     "skew after": round(player_dataset[n].dropna().skew(), 2)}
                    for n, (o, s, _) in CONV.items()])
display(_ct.sort_values("skew before", ascending=False))
_left = {c: round(player_dataset[c].dropna().skew(), 1)
         for c in player_dataset.select_dtypes("number")
         if c != "player_id" and player_dataset[c].dtype != "bool"
         and abs(player_dataset[c].dropna().skew()) > 3}
print("numeric features still |skew| > 3:", _left if _left else "none")
print("\nregistration_daypart:", player_dataset.registration_daypart.value_counts().to_dict())
print("registration_weekend: %.1f%%" % (player_dataset.registration_weekend.mean() * 100))

unused = [c for c in df.columns if c not in player_dataset.columns]
print(f"\nsource columns not carried ({len(unused)}):")
print(", ".join(unused))
print("\nDone.")

rows x cols      : (100000, 72)
unique columns   : True
inf values       : 0  (must be 0)

  log1p_login_frequency == log1p_login_rate               True
  log1p_deposits_per_month == log1p_deposit_rate          True
  log1p_bets_per_month == log1p_betting_rate              True
  log1p_betting_to_deposit_ratio == log1p_betting_intensity True
  expm1(monetary_scale) == total_deposits             True

missing % of derived columns:
log1p_win_rate                    42.16
slog1p_time_to_first_deposit      39.93
slog1p_GGR_per_day                 7.41
log1p_logins_per_active_day        7.41
log1p_betting_to_deposit_ratio     0.84
log1p_betting_intensity            0.84
log1p_winning_rate                 0.51
log1p_deposit_rate                 0.51
log1p_login_frequency              0.51
log1p_login_rate                   0.51
log1p_bets_per_month               0.51
log1p_betting_rate                 0.51
log1p_deposits_per_month           0.51
log1p_withdrawal_rate              0.51
monet

,new feature,converted from,skew before,skew after
13,log1p_win_rate,win_rate,134.2,1.05
6,log1p_betting_intensity,betting_intensity,58.5,0.59
15,log1p_betting_to_deposit_ratio,betting_to_deposit_ratio,58.5,0.59
7,log1p_total_wins,total_wins,30.6,0.58
14,log1p_winning_rate,winning_rate,26.9,0.75
5,log1p_total_withdrawals,total_withdrawals,26.6,0.97
12,log1p_betting_rate,betting_rate,26.3,0.67
18,log1p_bets_per_month,bets_per_month,26.3,0.67
9,slog1p_GGR,GGR,25.8,0.09
8,slog1p_net_deposits,net_deposits,23.6,0.15


numeric features still |skew| > 3: none

registration_daypart: {'night': 38918, 'evening': 27065, 'afternoon': 21314, 'morning': 12703}
registration_weekend: 30.1%

source columns not carried (33):
email, phone_number, status_reason, is_test_account, tc_accepted_date, first_name, middle_name, last_name, birth_date, employer_name, employer_city, employer_country, employer_postal_code, employer_phone_number, aml_risk_reason_code, industry, rg_risk_rating, btag, adfcd, ad_group, ad_set, marketing_creative, marketing_keyword, marketing_network, pep_type, utm_variant, marketing_click_date, marketing_location, total_deposits, total_withdrawals, total_bets, total_wins, total_logins

Done.
